# Manuscript exports

Rebuilds every figure and table of the manuscript from the analysis outputs, so nothing is edited by hand.
Run after notebooks 1 -> 1b -> 2 -> 3 -> 5. Writes to `Manuscript/Figures` and `Manuscript/Tables`.

| Output | Source |
|---|---|
| Figure 1 (flowchart) | `Manuscript/Figures/Figure 1 template.drawio` + counts printed by notebooks 1 and 2 |
| Figure 2 | `results/fig2_model_performance_shap_final.png` (notebook 3) |
| Figure 3 (sepsis KM) | `filtered_data/mimiciv_tabledata_imputed.csv` |
| Figure 4 (flow rate KM, matched) | `results/Flow_rate_matched_cohort.csv` (notebook 5) |
| Table 1 | `results/HFNOstudy_Table1_HFNOfailure_with_time.csv` (notebook 2) |
| Table 2 | `results/Table2_model_performance.csv`, `results/model_feature_counts.json` (notebook 3) |
| Tables A3, A4 | DeLong / ROX benchmark (notebook 3), flow-rate PSM (notebook 5) |

In [ ]:
import json, re, html, textwrap
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import openpyxl
from openpyxl.styles import Font, Alignment, Border, Side
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import xml.etree.ElementTree as ET

FIG_DIR, TAB_DIR = 'Manuscript/Figures', 'Manuscript/Tables'
DPI = 300
PINK, BLUE = '#E91E63', '#1E88E5'
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'pdf.fonttype': 42})

def save_tiff(fig_or_path, name):
    # Save as RGB TIFF with LZW compression at 300 dpi (plus a PNG preview).
    out = f'{FIG_DIR}/{name}'
    prev = f'{FIG_DIR}/_source_and_previews/{name}'   # PNG previews are kept out of the submission folder
    if isinstance(fig_or_path, str):
        im = Image.open(fig_or_path).convert('RGB')
    else:
        fig_or_path.savefig(prev + '.png', dpi=DPI, bbox_inches='tight', facecolor='white')
        im = Image.open(prev + '.png').convert('RGB')
    im.save(out + '.tiff', compression='tiff_lzw', dpi=(DPI, DPI))
    if isinstance(fig_or_path, str):
        im.save(prev + '.png', dpi=(DPI, DPI))
    print(f'Saved {out}.tiff  {im.size[0]}x{im.size[1]} px')

## Cohort counts (from the saved notebook outputs)

In [ ]:
def notebook_text(path):
    nb = json.load(open(path, encoding='utf-8'))
    out = []
    for c in nb['cells']:
        for o in c.get('outputs', []):
            t = o.get('text') or o.get('data', {}).get('text/plain', '')
            out.append(''.join(t))
    return '\n'.join(out)

def grab(text, pattern):
    m = re.findall(pattern, text)
    assert len(m) >= 1, pattern
    return int(m[-1])

t1, t2 = notebook_text('1_MIMICIV_HFNO_data_extraction.ipynb'), notebook_text('2_HFNO_data_processing.ipynb')
data = pd.read_csv('filtered_data/mimiciv_tabledata_imputed.csv', index_col=0)
counts = {
    'N_POOL':     grab(t1, r'pre-criteria pool\) : N = (\d+)'),
    'N_LT2H':     grab(t1, r'HFNO duration <2 hours\s+: N = (\d+)'),
    'N_GAP':      grab(t1, r'Time gap >4h to intubation\s+: N = (\d+)'),
    'N_LOS':      grab(t1, r'ICU stay <24 hours\s+: N = (\d+)'),
    'N_ICU1H':    grab(t1, r'HFNO <1 h after ICU admission\s+: N = (\d+)'),
    'N_DNI':      grab(t1, r'excluded due to status code: (\d+)'),
    'N_UNIQ':     grab(t1, r'Total unique criteria exclusions \(overlap possible\): N = (\d+)'),
    'N_AFTER':    grab(t1, r'After inclusion criteria \(before DNI/CMO\)\s+: N = (\d+)'),
    'N_CRITERIA': grab(t1, r'Number of patients remaining: (\d+)'),
    'N_MISSING':  grab(t2, r'Excluded patients \(>30% missing data\): (\d+)'),
    'N_ANALYSIS': len(data),
    'N_FAILURE':  int(data.hfno_failure.sum()),
}
counts['N_SUCCESS'] = counts['N_ANALYSIS'] - counts['N_FAILURE']
counts['N_EXCL1'] = counts['N_POOL'] - counts['N_CRITERIA']

# ---- reconcile every step of the flow, not just the missingness one ----------------
# The four inclusion criteria are counted independently in notebook 1's SQL
# (SUM(CASE WHEN ...) over the same pool), so a patient failing two is counted twice;
# N_UNIQ is the de-duplicated total. DNI/CMO is applied afterwards, to the N_AFTER
# survivors, so it cannot overlap with them.
_sep = counts['N_LT2H'] + counts['N_GAP'] + counts['N_LOS'] + counts['N_ICU1H']
assert counts['N_POOL'] - counts['N_UNIQ'] == counts['N_AFTER'], (
    f"pool {counts['N_POOL']} - unique exclusions {counts['N_UNIQ']} "
    f"!= after-criteria {counts['N_AFTER']}")
assert counts['N_AFTER'] - counts['N_DNI'] == counts['N_CRITERIA'], (
    f"after-criteria {counts['N_AFTER']} - DNI/CMO {counts['N_DNI']} "
    f"!= criteria-met {counts['N_CRITERIA']}")
assert counts['N_EXCL1'] == counts['N_UNIQ'] + counts['N_DNI'], (
    f"screening box total {counts['N_EXCL1']} != unique criteria {counts['N_UNIQ']} "
    f"+ DNI/CMO {counts['N_DNI']}")
assert _sep >= counts['N_UNIQ'], (
    f"the four criteria sum to {_sep}, fewer than the de-duplicated {counts['N_UNIQ']} "
    f"- they cannot overlap to a larger number")
assert counts['N_CRITERIA'] - counts['N_MISSING'] == counts['N_ANALYSIS']
assert counts['N_SUCCESS'] + counts['N_FAILURE'] == counts['N_ANALYSIS']

counts['N_OVERLAP'] = _sep - counts['N_UNIQ']

# ---- modelling band: one 85/15 split, reused by every landmark cohort ---------------
# Notebook 3 prints, per scenario s1..s5 (s1 = paper Model 5, s2 = Model 1, s3 = Model 2,
# s4 = Model 3, s5 = Model 4):
#     "sN: landmark  H h - patients P (failures F)"
#     "SN <name> - train: T, test: E, features: V"
t3 = notebook_text('3_HFNO_predictor_analysis.ipynb')

def _landmark(s):
    p, f = re.findall(rf's{s}: landmark\s+\d+ h - patients (\d+) \(failures (\d+)\)', t3)[-1]
    tr, te = re.findall(rf'S{s}\s.*?train:\s*(\d+),\s*test:\s*(\d+)', t3)[-1]
    return int(p), int(f), int(tr), int(te)

_S = {s: _landmark(s) for s in range(1, 6)}
for _s, (_p, _f, _tr, _te) in _S.items():
    assert _tr + _te == _p, f'scenario s{_s}: train {_tr} + test {_te} != cohort {_p}'

# scenario -> paper model number
_SC2PAPER = {1: 5, 2: 1, 3: 2, 4: 3, 5: 4}
counts['N_TRAIN'], counts['N_TEST'] = _S[1][2], _S[1][3]
assert _S[1][0] == counts['N_ANALYSIS'], 'scenario s1 cohort != analysed cohort'
assert counts['N_TRAIN'] + counts['N_TEST'] == counts['N_ANALYSIS']
assert _S[1][1] == counts['N_FAILURE'], 'scenario s1 failures != cohort failures'
# Models 1 and 2 share the 4 h landmark cohort
assert _S[2][:2] == _S[3][:2], 'Models 1 and 2 should share the 4 h landmark cohort'

for _tag, _s in (('LM4', 2), ('LM12', 4), ('LM24', 5)):
    _p, _f, _tr, _te = _S[_s]
    counts[f'{_tag}_N'], counts[f'{_tag}_F'] = _p, _f
    counts[f'{_tag}_TR'], counts[f'{_tag}_TE'] = _tr, _te
    assert _p <= counts['N_ANALYSIS'], f'{_tag} cohort exceeds the analysed cohort'
    assert _te <= counts['N_TEST'], (
        f'{_tag} test set ({_te}) exceeds the full test set ({counts["N_TEST"]}) - '
        'the landmark subsets must nest inside the single 85/15 split')

print('Modelling split reconciles:')
print(f"  {counts['N_ANALYSIS']} analysed -> development {counts['N_TRAIN']} "
      f"/ held-out test {counts['N_TEST']}")
for _tag, _h in (('LM4', 4), ('LM12', 12), ('LM24', 24)):
    print(f"  still on HFNO at {_h:>2} h: {counts[f'{_tag}_N']} "
          f"({counts[f'{_tag}_F']} failures) -> train {counts[f'{_tag}_TR']}, "
          f"test {counts[f'{_tag}_TE']}")
print(f"Figure 1 flow reconciles:")
print(f"  {counts['N_POOL']} pool")
print(f"  -{counts['N_UNIQ']} unique criteria exclusions "
      f"(bullets sum to {_sep}; {counts['N_OVERLAP']} patients meet >1 criterion)")
print(f"  ={counts['N_AFTER']} after inclusion criteria")
print(f"  -{counts['N_DNI']} DNI/CMO")
print(f"  ={counts['N_CRITERIA']} criteria met")
print(f"  -{counts['N_MISSING']} >30% missing")
print(f"  ={counts['N_ANALYSIS']} analysed = "
      f"{counts['N_SUCCESS']} success + {counts['N_FAILURE']} failure")
print(f"  screening box total = {counts['N_EXCL1']}")
counts

## Figure 1: flowchart (draw.io source is the template; this cell fills and renders it)

In [ ]:
src = open(f'{FIG_DIR}/_source_and_previews/Figure 1 template.drawio', encoding='utf-8').read()
for k, v in counts.items():
    src = src.replace(f'@@{k}@@', str(v))
assert '@@' not in src
open(f'{FIG_DIR}/_source_and_previews/Figure 1.drawio', 'w', encoding='utf-8').write(src)   # open in draw.io to edit by hand if needed

def style_dict(s):
    return dict(kv.split('=', 1) for kv in s.split(';') if '=' in kv)

def html_to_lines(value):
    # Return (header lines, bullet items) from draw.io HTML labels.
    v = re.sub(r'<li[^>]*>', '\x01', value)
    v = re.sub(r'<br[^>]*>|</div>|<div[^>]*>|</li>', '\n', v)
    v = html.unescape(re.sub(r'<[^>]+>', '', v))
    parts = v.split('\x01')
    head = [l.strip() for l in parts[0].split('\n') if l.strip()]
    bullets = [re.sub(r'\s+', ' ', p).strip() for p in parts[1:] if p.strip()]
    return head, bullets

root = ET.fromstring(src).find('root')
cells = {c.get('id'): c for c in root.findall('mxCell')}
xs, ys = [], []
for c in cells.values():
    g = c.find('mxGeometry')
    if c.get('vertex') and g is not None:
        x, y, w, h = (float(g.get(k, 0)) for k in ('x', 'y', 'width', 'height'))
        xs += [x, x + w]; ys += [y, y + h]
X0, X1, Y0, Y1 = min(xs) - 5, max(xs) + 5, min(ys) - 5, max(ys) + 5

fig = plt.figure(figsize=((X1 - X0) / 72, (Y1 - Y0) / 72))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(X0, X1); ax.set_ylim(Y1, Y0); ax.axis('off')
CH = 0.52   # average character width as a fraction of the font size (Helvetica)

def wrap(text, width_px, size):
    return textwrap.wrap(text, max(4, int(width_px / (CH * size))))

for c in cells.values():
    if not c.get('vertex'):
        continue
    g, st = c.find('mxGeometry'), style_dict(c.get('style', ''))
    x, y, w, h = (float(g.get(k, 0)) for k in ('x', 'y', 'width', 'height'))
    alpha = float(st.get('opacity', 100)) / 100
    ax.add_patch(Rectangle((x, y), w, h, facecolor=st.get('fillColor', 'white'), alpha=alpha, lw=0))
    ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=st.get('strokeColor', 'black'), lw=0.8))
    size = float(st.get('fontSize', 12))
    head, bullets = html_to_lines(c.get('value', ''))
    if st.get('textDirection') == 'vertical-rl':
        ax.text(x + w / 2, y + h / 2, ' '.join(head), rotation=90, ha='center', va='center', fontsize=size)
        continue
    if st.get('align') == 'left':
        lines, pad = [], float(st.get('spacingLeft', 5)) + 3
        for l in head:
            lines += [(0, s) for s in wrap(l, w - 2 * pad, size)]
        lines.append((0, ''))
        for b in bullets:
            wrapped = wrap(b, w - 2 * pad - 34, size)
            lines.append((12, '\u2022', wrapped[0]))
            lines += [(12, '', s) for s in wrapped[1:]]
        top = y + float(st.get('spacingTop', 5)) + 2
        for i, ln in enumerate(lines):
            yy = top + i * size * 1.2
            ax.text(x + pad + ln[0], yy, ln[1], ha='left', va='top', fontsize=size)
            if len(ln) == 3:
                ax.text(x + pad + ln[0] + 16, yy, ln[2], ha='left', va='top', fontsize=size)
    else:
        lines = []
        for l in head:
            lines += wrap(l, w - 12, size)
        y0 = y + h / 2 - (len(lines) - 1) * size * 1.2 / 2
        for i, s in enumerate(lines):
            ax.text(x + w / 2, y0 + i * size * 1.2, s, ha='center', va='center', fontsize=size)

def anchor(cell_id, fx, fy):
    g = cells[cell_id].find('mxGeometry')
    x, y, w, h = (float(g.get(k, 0)) for k in ('x', 'y', 'width', 'height'))
    return x + fx * w, y + fy * h

for c in cells.values():
    if not c.get('edge'):
        continue
    st, g = style_dict(c.get('style', '')), c.find('mxGeometry')
    pts = {p.get('as'): (float(p.get('x')), float(p.get('y'))) for p in g.findall('mxPoint') if p.get('as')}
    mids = [(float(p.get('x')), float(p.get('y'))) for p in g.findall('Array/mxPoint')]
    s = anchor(c.get('source'), float(st.get('exitX', .5)), float(st.get('exitY', 1))) if c.get('source') else pts['sourcePoint']
    t = anchor(c.get('target'), float(st.get('entryX', .5)), float(st.get('entryY', 0))) if c.get('target') else pts['targetPoint']
    if mids and st.get('elbow') == 'vertical':
        path = [s, (s[0], mids[0][1]), (t[0], mids[0][1]), t]
    elif abs(s[0] - t[0]) > 1 and abs(s[1] - t[1]) > 1:
        path = [s, (s[0], t[1]), t]
    else:
        path = [s, t]
    ax.plot([p[0] for p in path[:-1]] + [path[-1][0]], [p[1] for p in path[:-1]] + [path[-1][1]], color='black', lw=1)
    ax.annotate('', xy=path[-1], xytext=path[-2], arrowprops=dict(arrowstyle='-|>', color='black', lw=1, mutation_scale=14))

save_tiff(fig, 'Figure 1')
plt.show()

## Table 1

In [ ]:
t1 = pd.read_csv('results/HFNOstudy_Table1_HFNOfailure_with_time.csv', header=None, dtype=str).fillna('')
hdr = t1.iloc[1].tolist()                      # row 0: 'Grouped by ...', row 1: column names, row 2: n
col = {h: i for i, h in enumerate(hdr)}
N = int(float(t1.iloc[2, col['Overall']]))
n0, n1 = t1.iloc[2, col['0']], t1.iloc[2, col['1']]

V = 'Vital signs (24 hours before HFNO)'
L = 'Laboratory tests (last value before HFNO)'
H = 'HFNO parameters (first 2 hours of HFNO)'
ORDER = [  # (variable, manuscript label, section heading printed before it)
    ('gender', 'Gender, Male, n (%)', None), ('admission_age', 'Admission age, years', None),
    ('weight_admit', 'Weight, kg', None), ('race_cat', 'Ethnicity, n (%)', None),
    ('sepsis3', 'Sepsis, n (%)', 'Comorbidities'), ('myocardial_infarct', 'Myocardial infarct, n (%)', None),
    ('congestive_heart_failure', 'Congestive heart failure, n (%)', None),
    ('peripheral_vascular_disease', 'Peripheral vascular disease, n (%)', None),
    ('cerebrovascular_disease', 'Cerebrovascular disease, n (%)', None), ('chronic_pulmonary_disease', 'COPD, n (%)', None),
    ('liver_disease', 'Liver disease, n (%)', None), ('renal_disease', 'Renal disease, n (%)', None),
    ('malignant_cancer', 'Malignant cancer, n (%)', None), ('diabetes', 'Diabetes, n (%)', None),
    ('vp_last6h', 'Documented vasopressor use (last 6h), n (%)', None),
    ('gcs_binned', 'Glasgow Coma Score, n (%)', None), ('apsiii', 'APS-III score', None),
    ('heart_rate_mean_last24h', 'Heart rate, per minute', V), ('mbp_mean_last24h', 'MBP, mmHg', None),
    ('glucose_mean_last24h', 'Glucose, mg/dL', None), ('spo2_mean_last24h', 'SpO\u2082, %', None),
    ('spo2_last', 'SpO\u2082, last value before HFNO, %', None),
    ('resp_rate_mean_last24h', 'Respiratory rate, per minute', None),
    ('temperature_mean_last24h', 'Temperature, \u00baC', None), ('fluidbalance_24hr', 'Fluid balance, mL', None),
    ('platelet_last', 'Platelet, x 10\u2079/L', L), ('po2_last', 'pO\u2082, mmHg', None), ('pco2_last', 'pCO\u2082, mmHg', None),
    ('ptt_last', 'PTT, seconds', None), ('inr_last', 'INR', None), ('calcium_last', 'Calcium, mg/dL', None),
    ('potassium_last', 'Potassium, mmol/L', None), ('mchc_last', 'MCHC, g/dL', None), ('mch_last', 'MCH, pg', None),
    ('ph_last', 'pH', None), ('aniongap_last', 'Anion gap, mmol/L', None), ('sodium_last', 'Sodium, mmol/L', None),
    ('hemoglobin_last', 'Haemoglobin, g/dL', None), ('wbc_last', 'WBC, x 10\u2079/L', None), ('rdw_last', 'RDW, %', None),
    ('creatinine_last', 'Creatinine, mg/dL', None),
    ('flow_rate_2h', 'Flow rate, L/min', H), ('fio2_2h', 'FiO\u2082, %', None), ('rox_2h', 'ROX index at 2 hours', None),
    ('inhosp_mortality', 'In-hospital mortality, n (%)', 'Outcomes'),
    ('time_to_hfno', 'Time from hospital admission to HFNO, hours', None), ('time_on_hfno', 'HFNO duration, hours', None),
]

def base(label):
    return re.sub(r', (n \(%\)|mean \(SD\)|median \[Q1,Q3\])$', '', label)

variables = {}
for _, r in t1.iloc[3:].iterrows():
    variables.setdefault(base(r[0]), []).append(r)
missing_from_order = set(variables) - {v for v, _, _ in ORDER}
assert not missing_from_order, missing_from_order

rows = [[None, 'Missing, %', 'Overall', 'HFNO-success group', 'HFNO-failure group', 'P'],
        [None, None, None, f'(N={n0})', f'(N={n1})', None],
        ['N', None, N, int(n0), int(n1), None]]
section_rows = []
for var, label, section in ORDER:
    if section:
        rows.append([section, None, None, None, None, None]); section_rows.append(len(rows) - 1)
    rs = variables[var]
    miss = rs[0][col['Missing']]
    miss = f"{(float(miss) if miss else 0) / N * 100:.1f}%"
    pval = rs[0][col['P-Value']]
    if len(rs) == 1:
        r = rs[0]
        rows.append([label, miss, r[col['Overall']], r[col['0']], r[col['1']], pval])
    else:
        rows.append([label, miss, None, None, None, pval])
        for r in rs:
            level = r[1].title() if r[1].isupper() else r[1]
            rows.append([level or 'Missing', None, r[col['Overall']], r[col['0']], r[col['1']], None])

FOOT1 = ('Values are n (%), mean (SD) or median [Q1, Q3]. COPD: chronic obstructive pulmonary disease, APS-III: Acute '
         'Physiology Score III, MBP: mean arterial blood pressure, SpO\u2082: oxygen saturation, pO\u2082: partial pressure '
         'of oxygen, pCO\u2082: partial pressure of carbon dioxide, PTT: partial thromboplastin time, INR: international '
         'normalized ratio, MCH(C): mean corpuscular haemoglobin (concentration), WBC: white blood cell count, RDW: red '
         'blood cell distribution width, FiO\u2082: fraction of inspired oxygen, ROX: respiratory rate-oxygenation index.')

def write_xlsx(path, rows, footnote, bold_rows=(0,), italic_rows=(), bold_cells=(), widths=None):
    wb = openpyxl.Workbook(); ws = wb.active
    f = lambda **k: Font(name='Times New Roman', size=7, **k)
    for r in rows:
        ws.append(r)
    for i, row in enumerate(ws.iter_rows(min_row=1, max_row=len(rows))):
        for c in row:
            c.font = f(bold=(i in bold_rows) or ((i, c.column - 1) in bold_cells), italic=(i in italic_rows))
            c.alignment = Alignment(wrap_text=True, vertical='center')
    ws.append([footnote]); n = ws.max_row
    ws.merge_cells(start_row=n, start_column=1, end_row=n, end_column=len(rows[0]))
    ws.cell(n, 1).font = f(); ws.cell(n, 1).alignment = Alignment(wrap_text=True, vertical='top')
    ws.row_dimensions[n].height = 60
    for j, wdt in enumerate(widths or []):
        ws.column_dimensions[openpyxl.utils.get_column_letter(j + 1)].width = wdt
    thin = Side(style='thin')
    for c in ws[2]:
        c.border = Border(bottom=thin)
    wb.save(path); print('Saved', path)

write_xlsx(f'{TAB_DIR}/Table 1.xlsx', rows, FOOT1, bold_rows=[0] + section_rows, widths=[38, 10, 16, 16, 16, 8])
table1_rows = rows
pd.DataFrame(rows[3:], columns=['Variable'] + rows[0][1:]).head(12)

## Table 2 (manuscript numbering)

In [ ]:
t2 = pd.read_csv('results/Table2_model_performance.csv')
nfeat = json.load(open('results/model_feature_counts.json'))
# Notebook 3 already writes the manuscript numbering into Table2_model_performance.csv
# (Model 1 = 4h on-HFNO only ... Model 5 = Pre-HFNO only, counts 36/60/62/72/49).
# Remapping here would renumber a second time and scramble the rows.
NB_TO_PAPER = {f'Model {i}': f'Model {i}' for i in range(1, 6)}
DESC = {
    'Model 1': 'Model 1: Demographics, comorbidities, admission severity scores, 4-hour post-HFNO initiation data',
    'Model 2': 'Model 2: Demographics, comorbidities, admission severity scores, 24-hour pre-HFNO initiation data, 4-hour post-HFNO initiation data',
    'Model 3': 'Model 3: Demographics, comorbidities, admission severity scores, 24-hour pre-HFNO initiation data, 12-hour post-HFNO initiation data',
    'Model 4': 'Model 4: Demographics, comorbidities, admission severity scores, 24-hour pre-HFNO initiation data, 24-hour post-HFNO initiation data',
    'Model 5': 'Model 5: Demographics, comorbidities, admission severity scores, 24-hour pre-HFNO initiation data',
}
t2['paper'] = t2['Prediction model'].str.extract(r'^(Model \d)')[0].map(NB_TO_PAPER)
point = lambda s: float(s.split(' ')[0])
rows = [['Prediction model', 'Variables, n', 'Test patients (failures)', 'Classifier',
         'Sensitivity', 'Specificity', 'Accuracy', 'AUROC (95% CI)']]
bold = set()
for m in ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5']:
    g = t2[t2.paper == m]
    # Tripwire: the metrics come from the CSV via NB_TO_PAPER, but the variable count and
    # description are keyed by the paper label. If the two numberings ever drift apart the
    # row would show one model's count beside another model's metrics. The counts are
    # distinct (36/60/62/72/49), so comparing them catches exactly that.
    assert not g.empty, f'{m} missing from Table2_model_performance.csv'
    _csv_n = int(g['Variables, n'].iloc[0])
    assert _csv_n == nfeat[m], (
        f'{m}: Table2_model_performance.csv reports {_csv_n} variables but '
        f'model_feature_counts.json reports {nfeat[m]}. The model numbering in notebook 3 '
        f'and the manuscript numbering used here have drifted, so metrics and variable '
        f'counts would land on different rows. Check NB_TO_PAPER.')
    best = g['AUROC'].map(point).idxmax()
    for k, (i, r) in enumerate(g.iterrows()):
        rows.append([DESC[m] if k == 0 else None, nfeat[m] if k == 0 else None,
                     r['Test patients (failures)'] if k == 0 else None, r['Classifier'],
                     point(r['Sensitivity']), point(r['Specificity']), point(r['Accuracy']),
                     r['AUROC'].replace('\u2013', '\u2013')])
        if i == best:
            bold.add((len(rows) - 1, 7))
FOOT2 = ('Values for sensitivity, specificity, and accuracy are point estimates. AUROC is reported with 95% confidence '
         'intervals from a stratified percentile bootstrap over the test patients. All models include static baseline '
         'characteristics (demographics, comorbidities, and illness severity scores). Test sets differ between models '
         'because each model is applied only to the patients still receiving HFNO at the end of its window. Bold type '
         'indicates the highest AUROC within each prediction model; Abbreviations: AUROC, area under the receiver '
         'operating characteristic curve; HFNO, high-flow nasal oxygen.')
write_xlsx(f'{TAB_DIR}/Table 2.xlsx', rows, FOOT2, bold_cells=bold, widths=[45, 9, 14, 16, 10, 10, 10, 16])
table2_rows, table2_bold = rows, bold
pd.DataFrame(rows[1:], columns=rows[0])

## Figure 2

In [ ]:
Image.MAX_IMAGE_PIXELS = None
im = Image.open('results/fig2_model_performance_shap_final.png').convert('RGB')
im = im.resize((im.size[0] // 2, im.size[1] // 2), Image.LANCZOS)        # 600 -> 300 dpi
im.save(f'{FIG_DIR}/Figure 2.tiff', compression='tiff_lzw', dpi=(DPI, DPI))
im.save(f'{FIG_DIR}/_source_and_previews/Figure 2.png', dpi=(DPI, DPI))
print('Saved Figure 2', im.size)

## Figure 3: HFNO failure by sepsis

In [ ]:
d = data.copy()
start = pd.to_datetime(d.final_starttime)
hrs = lambda t: (pd.to_datetime(t) - start).dt.total_seconds() / 3600
d['time'] = np.where(d.hfno_failure == 1, np.where(d.intubated == 1, hrs(d.iv_time), hrs(d.deathtime)), hrs(d.final_endtime))
cph = CoxPHFitter().fit(d[['time', 'hfno_failure', 'sepsis3']], 'time', 'hfno_failure')
s = cph.summary.loc['sepsis3']
lr = logrank_test(d.time[d.sepsis3 == 1], d.time[d.sepsis3 == 0], d.hfno_failure[d.sepsis3 == 1], d.hfno_failure[d.sepsis3 == 0])
sepsis_hr = (s['exp(coef)'], s['exp(coef) lower 95%'], s['exp(coef) upper 95%'], lr.p_value)
print('Sepsis HR (unadjusted) %.2f (%.2f-%.2f), log-rank p=%.2g' % sepsis_hr)

def km_figure(groups, title_legend, hr, p, xmax=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    kmf = KaplanMeierFitter()
    for mask, color, label in groups:
        kmf.fit(d_km['time'][mask], d_km['E'][mask], label=label).plot_survival_function(ax=ax, color=color)
    ax.text(0.72, 0.30, f'HR = {hr[0]:.2f} ({hr[1]:.2f}\u2013{hr[2]:.2f})\n' + ('p < 0.001' if p < 0.001 else f'p = {p:.3f}'),
            transform=ax.transAxes, ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85, edgecolor='gray'))
    ax.set_xlabel('Time elapsed since HFNO initiation (hours)', fontsize=13, labelpad=8)
    ax.set_ylabel('Freedom from HFNO failure', fontsize=13, labelpad=8)
    ax.set_ylim(0, 1.02)
    if xmax:
        ax.set_xlim(0, xmax)
        # Tick every 24 h so the axis reads in days; matplotlib's default on a
        # 0-120 range is every 20 h, which has no clinical meaning.
        ax.set_xticks(range(0, int(xmax) + 1, 24))
    ax.tick_params(labelsize=11); ax.grid(True, linestyle='--', alpha=0.35)
    ax.legend(title=title_legend, loc='lower left', fontsize=11, title_fontsize=12, frameon=True)
    plt.tight_layout()
    return fig

XMAX = 120   # hours; few patients remain on HFNO beyond 5 days
d_km = pd.DataFrame({'time': d['time'], 'E': d.hfno_failure})
fig = km_figure([(d.sepsis3 == 0, BLUE, 'Other aetiologies'), (d.sepsis3 == 1, PINK, 'Sepsis (Sepsis-3 definition)')],
                'Aetiology', sepsis_hr[:3], sepsis_hr[3], XMAX)
save_tiff(fig, 'Figure 3'); plt.show()

## Figure 4: HFNO failure by mean flow rate over the first 2 h (propensity-matched)

In [ ]:
m = pd.read_csv('results/Flow_rate_matched_cohort.csv')
flow_res = pd.read_csv('results/Flow_rate_PSM_results.csv').set_index('Cohort')
thr = flow_res['threshold (L/min)'].iloc[0]
cphf = CoxPHFitter().fit(m[['time_fail_h', 'hfno_failure', 'high_flow']], 'time_fail_h', 'hfno_failure').summary.loc['high_flow']
flow_hr = (cphf['exp(coef)'], cphf['exp(coef) lower 95%'], cphf['exp(coef) upper 95%'], flow_res.loc['Propensity-matched', 'log-rank p'])
d_km = pd.DataFrame({'time': m.time_fail_h, 'E': m.hfno_failure})
fig = km_figure([(m.high_flow == 0, BLUE, f'< {thr:.0f} L/min'), (m.high_flow == 1, PINK, f'\u2265 {thr:.0f} L/min')],
                'Mean flow rate, first 2 h', flow_hr[:3], flow_hr[3], XMAX)
save_tiff(fig, 'Figure 4'); plt.show()

## Supplementary tables A3 (model comparisons and ROX benchmark) and A4 (flow-rate propensity matching)

In [ ]:
dl = pd.read_csv('results/Model_comparisons_DeLong.csv')
rox = pd.read_csv('results/TableS_ROX_benchmark.csv')
rows = [['Comparison', 'AUROC, model A', 'AUROC, model B', 'DeLong p']]
for _, r in dl.iterrows():
    rows.append([f"{r['Model A']} vs {r['Model B']}", round(r['AUROC A'], 2), round(r['AUROC B'], 2), round(r['DeLong p'], 3)])
rows.append(['ROX benchmark (same test patients with ROX available)', None, None, None])
for _, r in rox.iterrows():
    rows.append([f"{r['Predictor']} (available at: {r['Available at']})", r['AUROC (95% CI)'], None,
                 None if pd.isna(r['DeLong p vs ROX']) else round(r['DeLong p vs ROX'], 3)])
write_xlsx('results/Table A3.xlsx', rows,
           'Paired DeLong tests on the held-out test set (XGBoost). Model numbering as in the main text. ROX: '
           'respiratory rate-oxygenation index; AUROC: area under the receiver operating characteristic curve.',
           bold_rows=[0, len(dl) + 1], widths=[60, 18, 14, 10])

bal = pd.read_csv('results/Flow_rate_PSM_balance.csv', index_col=0)
fr = flow_res.reset_index()
rows = [['Cohort', 'n', 'Failures (high/low flow)', 'HR (95% CI)', 'Log-rank p']]
for _, r in fr.iterrows():
    rows.append([r['Cohort'], int(r['n']), r['failures high/low'], r['HR (95% CI)'], round(r['log-rank p'], 4)])
write_xlsx('results/Table A4.xlsx', rows,
           f'High flow: mean flow rate over the first 2 h \u2265 {thr:.0f} L/min (cohort median). Propensity score from all pre-initiation '
           f'variables; 1:1 nearest-neighbour matching, caliper 0.2 SD of the logit. Standardised mean differences: '
           f'{int((bal["SMD before"].abs() > 0.1).sum())} of {len(bal)} covariates > 0.1 before matching, '
           f'{int((bal["SMD after"].abs() > 0.1).sum())} after.', widths=[22, 8, 20, 18, 10])

In [ ]:
# Everything the manuscript text needs, in one place
summary = {
    'counts': counts,
    'sepsis_hr_unadjusted': sepsis_hr,
    'flow_threshold': thr, 'flow_hr_matched': flow_hr,
    'flow_pairs': int(len(m) / 2),
    'failure_deaths_without_intubation': int(((data.died == 1) & (data.intubated == 0)).sum()),
}
json.dump(summary, open('results/manuscript_numbers.json', 'w'), indent=1, default=float)
summary